# Building Features + Labels

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if not (Path.cwd() / "A_Data_Gathering").exists() else Path.cwd()

import pickle
import shutil
import time

import torch
from torch.utils.data import DataLoader

from C_Dataset_Building.helpers.feature_builder import precompute_and_cache, FEATURE_COLS
from C_Dataset_Building.helpers.dataset import StockDatasetSafe, is_cache_valid

root       = PROJECT_ROOT / "dataset"
stocks_dir = root / "stocks"
assert stocks_dir.exists(), f"Missing: {stocks_dir}"

files       = sorted(stocks_dir.glob("*.csv"))
cache_dir   = PROJECT_ROOT / "C_Dataset_Building" / f".feature_cache_forward_return_w{WINDOW}"
cache_dir.mkdir(parents=True, exist_ok=True)
scaler_path = cache_dir / "scaler.pkl"
index_path  = cache_dir / "index.pkl"

if REBUILD_FEATURE_CACHE:
    for p in [pp for pp in (PROJECT_ROOT / "C_Dataset_Building").glob(".feature*") if pp.exists()]:
        shutil.rmtree(p, ignore_errors=True) if p.is_dir() else p.unlink(missing_ok=True)
        print(f"[Cache] Removed: {p}")
    time.sleep(1)

cache_ready = is_cache_valid(scaler_path, index_path)
if not cache_ready and cache_dir.exists():
    print("[Cache] Stale / incomplete — wiping cache dir.")
    shutil.rmtree(cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_FEATURE_CACHE or not cache_ready:
    scaler, index = precompute_and_cache(
        files=files, window=WINDOW, cache_dir=cache_dir,
        scaler_path=scaler_path, index_path=index_path,
        horizon_bars=HORIZON_BARS, train_end_date=TRAIN_END_DATE,
        val_end_date=VAL_END_DATE, profit_threshold=PROFIT_THRESHOLD,
        stop_loss=STOP_LOSS,
    )
else:
    print("[Cache] Using existing feature cache.")
    with open(scaler_path, "rb") as f: scaler = pickle.load(f)
    with open(index_path,  "rb") as f: index  = pickle.load(f)

train_ds = StockDatasetSafe(index, scaler, "train")
val_ds   = StockDatasetSafe(index, scaler, "val")
test_ds  = StockDatasetSafe(index, scaler, "test")

_pin    = torch.cuda.is_available()
assert _pin, "GPU required but torch.cuda.is_available() is False."
_kwargs = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    pin_memory=_pin, persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)
train_loader = DataLoader(train_ds, shuffle=True,  **_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kwargs)

xb, yb = next(iter(train_loader))
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")
print(f"X batch: {xb.shape} {xb.dtype}  y batch: {yb.shape} {yb.dtype}")
print(f"Features: {len(FEATURE_COLS)} base x {WINDOW} lags = {len(FEATURE_COLS) * WINDOW}")